# 06 - Quick results within about 20 minutes

This recipe is for a first practical run. It keeps the modelling space small, proposes a modest number of models, and optionally estimates them with Apollo/R.

The exact runtime depends on the dataset, machine, and Apollo convergence.


## 1. Quick-run settings


In [7]:
import delphos as dp

RUN_ESTIMATION = True
OUTPUT_CSV = "quick_results.csv"

agent = dp.load_agent()
task = dp.load_dataset("Swissmetro")


## 2. Keep the search space compact

Start with transformations and tastes, but no covariates. This usually gives interpretable first models.


In [ ]:
quick_task = dp.configure_modelling_space(
    task,
    transformations=["linear", "log"],
    tastes=["generic", "specific"],
    covariates=[],
)

print("Attributes:", quick_task.attribute_names)
print("Transformations:", quick_task.transform_names)
print("Tastes:", quick_task.taste_names)
print("Covariates:", quick_task.covariate_names)


Attributes: ('ASC', 'time', 'cost', 'headway', 'seat')
Transformations: ('linear', 'log')
Tastes: ('generic', 'specific')
Covariates: ()


## 3. Generate a small candidate pool


In [9]:
models = agent.propose(
    quick_task,
    n_models=20,
    max_attempts=500,
    seed=2026,
)

df = models.to_dataframe()
df


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices
0,4,Swissmetro,1110_2210_3210_4120_5000_6220_7000,10,topk,0,False,None,5,"[25, 121, 225, 81, 33, 105, 113, 25, 65, 73]"
1,4,Swissmetro,1110_2220_3120_4210_5000_6110_7000,10,topk,1,False,None,5,"[17, 81, 25, 73, 33, 81, 113, 73, 65, 121]"
2,4,Swissmetro,1110_2220_3120_4210_5000_6220_7000,10,topk,2,False,None,5,"[73, 25, 81, 33, 225, 25, 113, 33, 65, 121]"
3,4,Swissmetro,1110_2220_3210_4210_5000_6220_7000,10,topk,3,False,None,5,"[17, 33, 25, 225, 17, 121, 25, 81, 33, 73]"
4,4,Swissmetro,1110_2220_3120_4110_5000_6110_7000,10,topk,5,False,None,5,"[17, 25, 225, 121, 33, 201, 81, 105, 73, 65]"
5,4,Swissmetro,1110_2220_3210_4110_5000_6110_7000,10,topk,6,False,None,5,"[73, 121, 25, 17, 33, 225, 25, 105, 201, 33]"
6,4,Swissmetro,1110_2120_3210_4110_5000_6220_7000,10,topk,7,False,None,5,"[25, 121, 33, 81, 105, 65, 225, 25, 17, 73]"
7,4,Swissmetro,1110_2210_3120_4110_5000_6110_7000,10,topk,8,False,None,5,"[25, 73, 17, 65, 81, 225, 33, 201, 65, 25]"
8,4,Swissmetro,1110_2220_3220_4120_5000_6220_7000,10,topk,12,False,None,5,"[25, 33, 121, 81, 17, 105, 25, 33, 113, 225]"
9,4,Swissmetro,1110_2120_3220_4110_5000_6110_7000,10,topk,14,False,None,5,"[25, 121, 17, 225, 81, 73, 105, 65, 81, 201]"


## 4. Optional estimation

For a quick first run, cap model complexity with `max_free_parameters`. This protects you from accidentally estimating very large models.


In [10]:
if RUN_ESTIMATION:
    models.estimate(
        quick_task,
        max_free_parameters=25,
        info=True,
        save=False,
    )
    df = models.to_dataframe()
else:
    print("Skipping Apollo/R estimation. Set RUN_ESTIMATION = True to estimate.")

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved results to {OUTPUT_CSV}")


2026-07-06 08:39:48,955 [INFO] Delphos.apollo: Starting Apollo estimation: 1110_2210_3210_4120_5000_6220_7000
Apollo ignition sequence completed
Several observations per individual detected based on the value of id.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Model run by gnova using Apollo 0.3.7 on R 4.5.1 for Darwin.
Please acknowledge the use of Apollo by citing Hess & Palma (2019)
  DOI 10.1016/j.jocm.2019.100170
  www.ApolloChoiceModelling.com

Model name                                  : 1110_2210_3210_4120_5000_6220_7000
Model description                           : MNL proposed by Delphos
Model run at                                : 2026-07-06 08:39:49.036085
Estimation method                           : bgw
Estimation diagnosis                        : Relative function convergence (favorable)
Optimisation diagnosis                      : Maximum found
     hessian properties                    

## 5. What to inspect first

After estimation, sort by reward or your custom score. Also inspect:

- `successfulEstimation`
- `LLout`
- `BIC`
- `nFreeParams`
- signs and magnitudes in Apollo output if saved
- repeated model families across strategies
